In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
import cartopy.crs as ccrs

In [ ]:
x_data = xr.open_dataset("../data/CanESM_1850-2100_rsutcs.nc", engine="netcdf4")
y_data = xr.open_dataset("../data/CanESM_1850-2100_rlutcs.nc", engine="netcdf4")

In [16]:
print(CanESM)
print("\nDimensions:", CanESM.dims)
print("Coordinates:", list(CanESM.coords))
print("Data variables:", list(CanESM.data_vars))

if "tas" in CanESM.data_vars:
    var_name = "tas"
else:
    # Fall back to the first non-bounds variable
    non_bnds = [v for v in CanESM.data_vars if not v.endswith("_bnds")]
    var_name = non_bnds[0] if non_bnds else list(CanESM.data_vars)[0]
da = CanESM[var_name]
print("\nSelected variable:", var_name)
print(da)

<xarray.Dataset> Size: 2GB
Dimensions:  (member: 25, time: 3012, lat: 64, lon: 128)
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    rsutcs   (member, time, lat, lon) float32 2GB ...

Dimensions: FrozenMappingWarningOnValuesAccess({'member': 25, 'time': 3012, 'lat': 64, 'lon': 128})
Coordinates: ['time', 'lat', 'lon', 'member']
Data variables: ['rsutcs']

Selected variable: rsutcs
<xarray.DataArray 'rsutcs' (member: 25, time: 3012, lat: 64, lon: 128)> Size: 2GB
[616857600 values with dtype=float32]
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 5

In [18]:
print(CanESM.dims)        # dimension names + sizes
print(CanESM.coords)      # coordinate variables
print(CanESM.data_vars)   # actual data variables
print(CanESM.attrs)       # global attributes

print(CanESM.variables)
# var = CanESM[var_name]
# print(var.dimensions)
# print(var.shape)
# print(var.ncattrs())

FrozenMappingWarningOnValuesAccess({'member': 25, 'time': 3012, 'lat': 64, 'lon': 128})
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    rsutcs   (member, time, lat, lon) float32 2GB ...
{}
Frozen({'rsutcs': <xarray.Variable (member: 25, time: 3012, lat: 64, lon: 128)> Size: 2GB
[616857600 values with dtype=float32]
Attributes:
    standard_name:  toa_outgoing_shortwave_flux_assuming_clear_sky
    long_name:      TOA Outgoing Clear-Sky Shortwave Radiation
    comment:        Calculated in the absence of clouds.
    units:          W m-2
    original_name:  FSRC
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    history:        2019-07-06T01:28:30Z altered by CMOR: R

In [ ]:
CanESM = xr.open_dataset("../data/CanESM_1850-2100_rsutcs.nc", engine="netcdf4")
tas_var = "tas" if "tas" in CanESM.data_vars else list(CanESM.data_vars)[0]
tas_da = CanESM[tas_var]
print("tas variable:", tas_var)

rsutcs_var = "rsutcs" if "rsutcs" in CanESM.data_vars else list(CanESM.data_vars)[0]
rsutcs_da = CanESM[rsutcs_var]
rsutcs_da, tas_da = xr.align(rsutcs_da, tas_da, join="inner")

rng = np.random.default_rng(42)
time_idx = np.arange(rsutcs_da.sizes["time"])
rng.shuffle(time_idx)

n = time_idx.size
n_train = int(n * 0.8)
n_val = int(n * 0.1)

train_idx = time_idx[:n_train]
val_idx = time_idx[n_train:n_train + n_val]
test_idx = time_idx[n_train + n_val:]

X_train = tas_da.isel(time=train_idx)
y_train = rsutcs_da.isel(time=train_idx)
X_val = tas_da.isel(time=val_idx)
y_val = rsutcs_da.isel(time=val_idx)
X_test = tas_da.isel(time=test_idx)
y_test = rsutcs_da.isel(time=test_idx)

print("train/val/test sizes:", X_train.sizes["time"], X_val.sizes["time"], X_test.sizes["time"])

tas variable: rsutcs
train/val/test sizes: 2409 301 302
